# Clase 069 — Regresión lineal: ecuación normal vs gradient descent

Resolvemos regresión lineal de **dos formas** y verificamos que dan el mismo óptimo: la **ecuación normal** cerrada $\hat\theta = (X^T X)^{-1} X^T y$ y el **gradient descent** iterativo. Comparamos también contra `LinearRegression` de sklearn (pseudoinversa SVD) y medimos cómo escala el coste con el número de features.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset sintético

Generamos $y = 4 + 3x + \text{ruido}\ \mathcal{N}(0,1)$ con $m=100$. El $\theta$ verdadero es $[4, 3]$, así que podremos verificar cuánto lo recupera cada método. Agregamos la columna de unos $x_0=1$ para absorber el bias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 100
X = 2 * np.random.rand(m, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(m)     # DGP: y = 4 + 3x + N(0,1)
X_b = np.c_[np.ones((m, 1)), X]              # columna de unos para el bias
print('X_b shape:', X_b.shape, '| y shape:', y.shape)
print('theta verdadero:', [4, 3])

## 2. Ecuación normal a mano (NumPy)

Igualando el gradiente del MSE a cero sale la solución cerrada $\hat\theta = (X^T X)^{-1} X^T y$. La resolvemos con `np.linalg.inv` y el operador `@`.

In [ ]:
theta_normal = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print('theta (ecuación normal):', theta_normal.round(3))
assert np.allclose(theta_normal, [4, 3], atol=0.5), 'debería recuperar ~[4, 3]'
print('OK: recupera el theta verdadero [4, 3]')

## 3. Pseudoinversa de Moore-Penrose (SVD)

La pseudoinversa $X^+ = V\Sigma^+U^T$ **siempre existe** y es más estable que invertir $X^T X$. En un caso bien condicionado debe dar exactamente lo mismo que la ecuación normal.

In [ ]:
theta_pinv = np.linalg.pinv(X_b) @ y
print('theta (pseudoinversa SVD):', theta_pinv.round(3))
assert np.allclose(theta_normal, theta_pinv, atol=1e-6), 'deben coincidir'
print('OK: ecuación normal y pseudoinversa coinciden en caso bien-condicionado')

## 4. `LinearRegression` de sklearn

sklearn **no** usa la ecuación normal: llama a `scipy.linalg.lstsq` (SVD). Sus `intercept_` y `coef_` deben coincidir con nuestro cálculo manual.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(X, y)
print('intercept_:', round(lin.intercept_, 3), '| coef_:', lin.coef_.round(3))
assert abs(lin.intercept_ - theta_normal[0]) < 1e-6
assert abs(lin.coef_[0] - theta_normal[1]) < 1e-6
print('predicción en x=0 y x=2:', lin.predict([[0], [2]]).round(3))
print('OK: sklearn coincide con la ecuación normal manual')

## 5. Caso singular: features colineales

Si $x_2 = 2x_1$, la matriz $X^T X$ es **singular**: `np.linalg.inv` falla o devuelve basura. La pseudoinversa sí funciona (reparte el peso entre las columnas colineales) y las predicciones son idénticas al modelo bien condicionado, porque el espacio columna es el mismo.

In [ ]:
X_col = np.c_[X, 2 * X]                       # x2 = 2*x1  -> colineal
Xc_b = np.c_[np.ones((m, 1)), X_col]
try:
    theta_bad = np.linalg.inv(Xc_b.T @ Xc_b) @ Xc_b.T @ y
    print('inv devolvió (posible basura):', theta_bad.round(2))
except np.linalg.LinAlgError as e:
    print('inv falló con LinAlgError:', e)

theta_ok = np.linalg.pinv(Xc_b) @ y
print('pinv sí funciona:', theta_ok.round(3))
assert np.allclose(Xc_b @ theta_ok, X_b @ theta_normal, atol=1e-6)
print('OK: pinv da las mismas predicciones pese a la colinealidad')

## 6. Gradient descent (batch) a mano

Arrancamos con $\theta$ aleatorio y restamos $\eta\cdot\nabla_\theta\text{MSE}$, con $\nabla = \frac{2}{m}X^T(X\theta - y)$. Tras suficientes iteraciones converge a la solución cerrada.

In [ ]:
eta, n_iter = 0.1, 1000
rng = np.random.default_rng(0)
theta_gd = rng.standard_normal(2)
cost_history = []
for _ in range(n_iter):
    grad = (2 / m) * X_b.T @ (X_b @ theta_gd - y)
    theta_gd -= eta * grad
    cost_history.append(np.mean((X_b @ theta_gd - y) ** 2))

print('theta (gradient descent):', theta_gd.round(3))
assert np.allclose(theta_gd, theta_normal, atol=1e-2), 'GD converge a la forma cerrada'
print('OK: GD converge al mismo theta que la ecuación normal')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cost_history)
ax.set_xlabel('iteración'); ax.set_ylabel('MSE')
ax.set_title('Convergencia del gradient descent (batch)')
plt.tight_layout(); plt.show()

## 7. Complejidad empírica en features

Invertir $X^T X$ cuesta $O(n^3)$ en features. Cronometramos `LinearRegression().fit` con un número creciente de columnas y $m$ fijo: el tiempo crece rápido.

In [ ]:
import time
from sklearn.datasets import make_regression

sizes, times = [50, 200, 500, 1000], []
for n in sizes:
    Xn, yn = make_regression(n_samples=2000, n_features=n, noise=10, random_state=42)
    t0 = time.perf_counter()
    LinearRegression().fit(Xn, yn)
    times.append(time.perf_counter() - t0)
    print(f'n_features={n:5d} -> {times[-1]*1000:7.1f} ms')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, times, 'o-')
ax.set_xlabel('nº de features'); ax.set_ylabel('tiempo de fit (s)')
ax.set_title('El coste de la forma cerrada crece con las features')
plt.tight_layout(); plt.show()

## Ejercicios

1. **Pseudoinversa vs inversa.** Con $n > m$ (más features que muestras), construí $X$ y comprobá que `np.linalg.inv(X_b.T @ X_b)` falla mientras `np.linalg.pinv(X_b) @ y` devuelve la solución de norma mínima.
2. **Sin la columna de unos.** Resolvé la ecuación normal olvidando `X_b` (sin bias) y verificá que el modelo pasa forzado por el origen: el intercept desaparece.
3. **Escalado y `coef_`.** Multiplicá `X` por 1000 y reajustá. La predicción no cambia, pero `coef_` sí: discutí por qué solo son interpretables con features en la misma escala.
4. **Ecuación normal manual sobre `load_diabetes`.** Escalá las features, resolvé $\hat\theta$ a mano y verificá que coincide con `LinearRegression` con tolerancia `1e-6`.

## Conclusiones

- La ecuación normal, la pseudoinversa SVD, sklearn y el gradient descent convergen todos al **mismo óptimo** en un problema bien condicionado.
- La pseudoinversa es la opción robusta: funciona aunque $X^T X$ sea singular (features colineales o $n > m$).
- La forma cerrada escala mal en **features** ($O(n^3)$) pero bien en muestras; con muchas columnas conviene gradient descent.
- Escalar las features no cambia la predicción de la forma cerrada, pero es imprescindible para interpretar `coef_` y para que GD converja rápido.